In [1]:
pip install torch transformers huggingface_hub matplotlib seaborn numpy accelerate protobuf tiktoken sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 338.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 382.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Repetition as Hiroka (2024) work

We use this code to generate repetition dataset to identify repetition neurons in text generation task

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import regex as re
import json
import argparse
import torch
def countOverlap(text, query):
    return len(re.findall(query, text, overlapped=True))

def getFirstAppearingIdx(text, query):
    return re.finditer(query, text).__next__().start(0)


def detectRepetition(line, n, r, k):
    # line should be list of idx
    SEP = ' '
    for i in range(0, len(line)-n+1):
        ngram = line[i:i+n]
        ngramStr = SEP.join(map(str, ngram))
        lineRange = line[max(0, i+n-r):i+n]
        lineRangeStr = SEP.join(map(str, lineRange))
        countRepInRange = countOverlap(lineRangeStr, ngramStr)

        if k <= countRepInRange:
            try:
                firstPosition = getFirstAppearingIdx(lineRangeStr, ngramStr)
                firstPosition = len(lineRangeStr[:firstPosition].split()) # クエリにスペースを含むので，分割し直してから位置を再計算
                firstPosition += max(0, i+n-r)

                lineRangeStr4second = SEP.join(map(str, line[firstPosition+1:i+n]))
                secondPosition = getFirstAppearingIdx(lineRangeStr4second, ngramStr)
                secondPosition = len(lineRangeStr4second[:secondPosition].split())
                secondPosition += firstPosition + 1

                lineRangeStr4third = SEP.join(map(str, line[secondPosition+1:i+n]))
                thirdPosition = getFirstAppearingIdx(lineRangeStr4third, ngramStr)
                thirdPosition = len(lineRangeStr4third[:thirdPosition].split())
                thirdPosition += secondPosition + 1
                if (secondPosition - firstPosition) == (thirdPosition - secondPosition):
                    return ngram, firstPosition, secondPosition, thirdPosition
            except:
                pass
    return [], -1, -1, -1

def generateDataSample(model, tokenizer, n=10, r=100, k=3, minimumNonRepetitiveAffix=50, numRansomSampleToekns=10, numGreedyGenerationTokens=200):
    # set generation config
    generationConfigSample = GenerationConfig(max_new_tokens=numRansomSampleToekns, do_sample=True, 
                                              eos_token_id=model.config.eos_token_id, temperature=1.0, pad_token_id=tokenizer.pad_token_id,)
    generationConfigGreedy = GenerationConfig(max_new_tokens=numGreedyGenerationTokens, do_sample=False, 
                                              eos_token_id=model.config.eos_token_id, pad_token_id=tokenizer.pad_token_id,)

    # sampling first {numRansomSampleToekns} tokens
    initialInput = tokenizer(' ', return_tensors="pt", padding=True)
    initialInput = {k: v.to(model.device) for k, v in initialInput.items()}
    initialOutputs = model.generate(
    input_ids=initialInput["input_ids"],
    attention_mask=initialInput["attention_mask"],
    generation_config=generationConfigSample,
    pad_token_id=tokenizer.pad_token_id,
)
    #print('init:', repr(tokenizer.decode(initialOutputs[0])))

    # greedy generation for next {numGreedyGenerationTokens} tokens
    additionalOutputs = model.generate(initialOutputs, generation_config=generationConfigGreedy)
    #print(' add:', repr(tokenizer.decode(additionalOutputs[0])))

    ngram, firstPosition, secondPosition, thirdPosition = detectRepetition(additionalOutputs[0].tolist(), n, r, k)
    if ngram and minimumNonRepetitiveAffix < secondPosition:
        dumpLine = {
            'promptIds': initialOutputs[0].tolist(),
            'promptTokens': tokenizer.decode(initialOutputs[0]),
            'firstPosition': firstPosition,
            'secondPosition': secondPosition,
            'thirdPosition': thirdPosition,
            'ngramIds': ngram,
            'ngramTokens': tokenizer.decode(ngram),
            #"generatedIds":additionalOutputs[0].tolist(),
            '50TokensBeforeRepeat': tokenizer.decode(additionalOutputs[0].tolist()[secondPosition-50:secondPosition]),
            '50TokensAfterRepeat': tokenizer.decode(additionalOutputs[0].tolist()[secondPosition:secondPosition+50]),
            'generatedIds': additionalOutputs[0].tolist()
        }
        return dumpLine
        
    
    return None



In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
from huggingface_hub import login
import json
login(token='put your token here')#
# -----------------------------
# 1. Load the Model and Tokenizer
# -----------------------------
#model_name = 'meta-llama/Llama-2-13b-hf'#"meta-llama/Llama-3.1-8B" 

model_name = 'meta-llama/Llama-3.1-8B'#"meta-llama/Llama-2-13b-hf"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_auth_token=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    use_auth_token=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval().to("cuda")
def seed_everything(seed: int):
    import random, os
    import numpy as np
    import torch

    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
seed_everything(42)


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/tokenization_auto.py:898: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py:476: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [9]:
from tqdm import tqdm
import os
BASE_DIR = os.getcwd()
filename = f"repetition_neuron_data/{model_name[-10:]}.jsonl"
output_path = os.path.join(BASE_DIR, filename)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
for i in tqdm(range(1000)):
    dumpLine = generateDataSample(model, tokenizer)
    if dumpLine:
        with open(output_path, 'a') as f:
            f.write(json.dumps(dumpLine)+'\n')

  7%|▋         | 74/1000 [08:28<1:46:01,  6.87s/it]


KeyboardInterrupt: 